# 02 - Validación del pipeline ElectroCasa

Última tarea del job. Hace tres cosas:
1. Muestra los conteos de cada tabla de Silver y Gold.
2. Muestra la cantidad de registros rechazados por fuente.
3. **Falla** si alguna tabla de Gold está vacía (o no existe).

Parámetro: `catalog` (se pasa desde el job con `${var.catalog}`).

In [0]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo")
catalog = dbutils.widgets.get("catalog")
print(f"Catálogo validado: {catalog}")

# Tablas esperadas por capa (ajusta si cambiaste algún nombre)
SILVER = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales", "tracking_envios"]
GOLD = ["ventas_netas", "top_productos_cantidad", "top_productos_devolucion",
        "dotacion_empleados", "ventas_vendedor", "resenas", "despacho", "dim_empleados"]

# Fuentes con su tabla Bronze y Silver (para estimar rechazados)
FUENTES = {
    "ventas": ("ventas", "ventas"),
    "catalogo": ("catalogo", "catalogo"),
    "devoluciones": ("devoluciones", "devoluciones"),
    "resenas": ("resenas", "resenas"),
    "sucursales": ("sucursales", "sucursales"),
    "tracking_envios": ("tracking_envios", "tracking_envios"),
}

In [0]:
def contar(schema, tabla):
    """Devuelve el conteo de filas, o None si la tabla no existe / no se puede leer."""
    try:
        return spark.table(f"{catalog}.{schema}.{tabla}").count()
    except Exception as e:
        print(f"[AVISO] No se pudo leer {catalog}.{schema}.{tabla}: {str(e)[:150]}")
        return None

## 1. Conteos de Silver y Gold

In [0]:
filas = []
for t in SILVER:
    filas.append(("silver", t, contar("silver", t)))
for t in GOLD:
    filas.append(("gold", t, contar("gold", t)))

df_conteos = spark.createDataFrame(filas, "capa string, tabla string, filas long")
display(df_conteos.orderBy("capa", "tabla"))

## 2. Registros rechazados por fuente

Si existe una tabla de cuarentena (`silver.cuarentena`), se usa como fuente de verdad.
Si no, se **estima** como `Bronze - Silver` (aproximación: también incluye reingestas deduplicadas,
y no aplica a tablas con SCD Type 2 como Empleados, donde Silver guarda versiones).

In [0]:
hay_cuarentena = spark.catalog.tableExists(f"{catalog}.silver.cuarentena")

if hay_cuarentena:
    print("Usando silver.cuarentena como fuente de rechazados")
    try:
        df_rech = spark.sql(f"""
            SELECT fuente, COUNT(*) AS rechazados
            FROM {catalog}.silver.cuarentena
            GROUP BY fuente
            ORDER BY fuente
        """)
        display(df_rech)
    except Exception as e:
        print(f"[AVISO] cuarentena existe pero no tiene la columna 'fuente': {str(e)[:150]}")
        hay_cuarentena = False

if not hay_cuarentena:
    print("Estimando rechazados como Bronze - Silver")
    est = []
    for fuente, (b, s) in FUENTES.items():
        nb = contar("bronze", b)
        ns = contar("silver", s)
        r = (nb - ns) if (nb is not None and ns is not None) else None
        est.append((fuente, nb, ns, r))
    display(spark.createDataFrame(est, "fuente string, bronze long, silver long, rechazados_estimados long"))
    print("Nota: Empleados se omite (SCD Type 2: Silver guarda versiones, no filas 1 a 1).")

## 3. Validación final (falla el job si Gold está vacío)

In [0]:
problemas = []
for t in GOLD:
    n = contar("gold", t)
    if n is None:
        problemas.append(f"gold.{t}: no existe o no se pudo leer")
    elif n == 0:
        problemas.append(f"gold.{t}: está vacía")

if problemas:
    detalle = "\n - ".join(problemas)
    raise Exception(f"VALIDACIÓN FALLIDA en {catalog}:\n - {detalle}")

print(f"VALIDACIÓN OK: las {len(GOLD)} tablas de Gold tienen datos en {catalog}.")